# SFT Notebook - Supervised Fine-Tuning

**Enhanced with W&B monitoring and validation metrics**

## Overview

This notebook performs Supervised Fine-Tuning (SFT) on a CPT-enhanced model for DFK classification.

### Key Features

- **Real-time W&B monitoring** - Watch training progress live in browser
- **Incremental data support** - Add new CSV, rerun, model continues learning
- **Automatic CPT LoRA loading** - Builds on CPT output automatically
- **Validation metrics tracking** - Monitors accuracy, F1, precision, recall
- **Label imbalance detection** - Shows distribution of DFK categories

## Configuration Section

In [ ]:
# ============================================
# CONFIGURATION - EDIT THIS SECTION
# ============================================

# Base model (Unsloth pre-quantized 4-bit)
MODEL_NAME = "unsloth/Qwen3-8B-bnb-4bit"

# W&B Configuration (for real-time monitoring)
WANDB_PROJECT = "tim1-dfk"
WANDB_RUN_NAME = "sft-run-"  # Timestamp will be added automatically
WANDB_ENTITY = None  # Set to your W&B username if not using default

# Training Configuration
NUM_EPOCHS = 5
BATCH_SIZE = 8  # Effective batch size
LEARNING_RATE = 2e-4
VALIDATION_SPLIT = 0.1  # 10% for validation

# Use existing CPT LoRA if found?
USE_CPT_LORA = True

# Run inference test after training?
RUN_INFERENCE_TEST = True

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
print(f"  Model: {MODEL_NAME}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Validation Split: {VALIDATION_SPLIT * 100}%")
print(f"  Use CPT LoRA: {USE_CPT_LORA}")
print(f"  Inference Test: {RUN_INFERENCE_TEST}")
print(f"  W&B Project: {WANDB_PROJECT}")
print("=" * 60)
print("")

## Step 1: Setup Environment

In [ ]:
# Setup environment
import os
import sys
from pathlib import Path
from datetime import datetime

# Set HF cache to local disk (faster than Google Drive)
if 'COLAB_GPU' in os.environ:
    os.environ['HF_HOME'] = '/content/hf_cache'
    os.makedirs('/content/hf_cache', exist_ok=True)
    os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')

# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Navigate to project directory
DRIVE_PATH = '/content/drive/MyDrive/Tim1-DFK'
if not os.path.exists(DRIVE_PATH):
    print('ERROR: Tim1-DFK folder not found on Google Drive!')
    print('Please upload your project folder to Google Drive first.')
    sys.exit(1)

os.chdir(DRIVE_PATH)
print(f'Working directory: {os.getcwd()}')
print(f'Timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

## Step 2: Setup W&B (Weights & Biases)

**W&B provides real-time visualization of training progress and validation metrics.**

Benefits:
- Live loss curves in browser
- Compare multiple experiments
- Track hyperparameters
- Monitor validation accuracy, F1, precision, recall

In [ ]:
# Setup W&B for real-time training monitoring
print('\n' + '=' * 60)
print('SETTING UP W&B MONITORING')
print('=' * 60)

try:
    import wandb

    # Login to W&B (will prompt for API key if not already logged in)
    wandb.login()

    # Verify connection
    api = wandb.Api()
    viewer = api.viewer
    username = viewer.entity if hasattr(viewer, 'entity') else str(viewer)
    print(f'✅ W&B connected as: {username}')
    print(f"   Project: {WANDB_PROJECT}")

    # Set W&B API key as environment variable for scripts
    os.environ['WANDB_API_KEY'] = wandb.api.api_key
    if WANDB_ENTITY:
        os.environ['WANDB_ENTITY'] = WANDB_ENTITY

    # Generate unique run name with timestamp
    timestamp = datetime.now().strftime("%m%d_%H%M%S")
    WANDB_RUN_FULL = f"{WANDB_RUN_NAME}{timestamp}"
    os.environ['WANDB_RUN_NAME'] = WANDB_RUN_FULL

    print(f"   Run name: {WANDB_RUN_FULL}")
    entity_display = WANDB_ENTITY or username
    print(f"   Monitor at: https://wandb.ai/{entity_display}/{WANDB_PROJECT}")

    print('')
    print('Metrics to monitor:')
    print('  - train/loss: Training loss over time')
    print('  - eval/loss: Validation loss')
    print('  - eval/accuracy: Classification accuracy')
    print('  - eval/f1_macro: Macro F1 score (handles imbalance)')
    print('  - train/learning_rate: Learning rate schedule')
    print('')
    print('What to look for:')
    print('  ✓ eval/loss < train/loss (not overfitting)')
    print('  ✓ eval/accuracy increasing (model learning)')
    print('  ✓ No sudden spikes in loss')

except ImportError:
    print('')
    print('⚠️  W&B not installed.')
    print('   Training will run without monitoring.')
    print('   To install, run: pip install wandb')
    os.environ['WANDB_DISABLED'] = 'true'

print('')
print('=' * 60)

## Step 3: Check Model & CPT LoRA

In [ ]:
# Check model cache and CPT LoRA
print('\n' + '=' * 60)
print('CHECKING MODEL & CPT LORA STATUS')
print('=' * 60)

# Check model cache (generic path from model name)
MODEL_CACHE = Path('/content/hf_cache/' + MODEL_NAME.split('/')[-1])

if MODEL_CACHE.exists():
    files = list(MODEL_CACHE.rglob('*'))
    total_size = sum(f.stat().st_size for f in files if f.is_file()) / 1e9
    print(f'✅ Model cached locally')
    print(f'   Path: {MODEL_CACHE}')
    print(f'   Size: {total_size:.2f} GB')
    print(f'   Files: {len(files)}')
    MODEL_CACHED = True
else:
    print(f'ℹ️  Model not cached yet')
    print(f'   Will download from: {MODEL_NAME}')
    print(f'   Estimated download: ~5 GB (pre-quantized 4-bit)')
    MODEL_CACHED = False

print('')
# Check CPT LoRA
CPT_LORA_PATH = Path('outputs/cpt/lora_adapter')

if USE_CPT_LORA and CPT_LORA_PATH.exists():
    files = list(CPT_LORA_PATH.rglob('*'))
    total_size = sum(f.stat().st_size for f in files if f.is_file()) / (1024*1024)
    print(f'✅ CPT LoRA adapter found!')
    print(f'   Location: {CPT_LORA_PATH}')
    print(f'   Size: {total_size:.1f} MB')
    print(f'   Files: {len(files)}')
    print('')
    print('🔄 CPT + SFT Mode ACTIVE:')
    print('   ✓ Will load CPT LoRA first')
    print('   ✓ Then train SFT on top')
    print('   ✓ Final model has CPT + SFT knowledge')
    CPT_AVAILABLE = True
elif USE_CPT_LORA:
    print('')
    print('⚠️  CPT LoRA not found, but USE_CPT_LORA = True')
    print('   SFT will attempt to load CPT LoRA')
    print('   For best results, run 1_CPT_Colab.ipynb first')
    CPT_AVAILABLE = False
else:
    print('')
    print('ℹ️  CPT LoRA will NOT be loaded')
    print('   SFT will train from base model only')
    CPT_AVAILABLE = False

print('')
print('=' * 60)
print('READY FOR SFT TRAINING')
print('=' * 60)
print('')
print(f"  Base Model: {MODEL_NAME}")
print(f"  CPT LoRA: {'Yes (will load)' if CPT_AVAILABLE or USE_CPT_LORA else 'No (training from base)'}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Validation: {VALIDATION_SPLIT * 100}% split")
print(f"  W&B Project: {WANDB_PROJECT}")

## Step 4: Check SFT Data & Label Distribution

In [ ]:
# Check SFT data files and analyze label distribution
print('\n' + '=' * 60)
print('CHECKING SFT DATA')
print('=' * 60)

SFT_DIR = Path('Dataset/SFT')
csv_files = list(SFT_DIR.glob('*.csv')) + list(SFT_DIR.glob('*.xlsx'))
processed_dir = SFT_DIR / 'processed'
train_file = processed_dir / 'sft_train_alpaca.json'
val_file = processed_dir / 'sft_val_alpaca.json'
stats_file = processed_dir / 'sft_stats.json'

print(f'Available SFT Data Files: {len(csv_files)}')
for f in csv_files:
    size_mb = f.stat().st_size / (1024*1024)
    print(f'  - {f.name} ({size_mb:.1f} MB)')

print('')
if stats_file.exists():
    import json
    with open(stats_file) as f:
        stats = json.load(f)
    
    total_entries = stats['total_entries']
    train_size = stats['train_size']
    val_size = stats['val_size']
    label_dist = stats['label_distribution']
    
    print(f'✅ Preprocessed data found!')
    print(f'   Total entries: {total_entries:,}')
    print(f'   Train samples: {train_size:,}')
    print(f'   Val samples: {val_size:,}')
    print(f'')
    print('Label Distribution:')
    print('   Label' + ' ' * 15 + 'Count' + ' ' * 10 + 'Percentage')
    print('   ' + '-' * 50)
    
    for label, count in sorted(label_dist.items(), key=lambda x: -x[1]):
        pct = count / total_entries * 100
        bar = '█' * int(pct / 2)
        bar = bar.ljust(50, '░')
        print(f'   {label.ljust(15)} {count:>8,} {bar} {pct:5.1f}%')
    
    print('   ' + '-' * 50)
    
    # Check for label imbalance warnings
    min_count = min(label_dist.values())
    max_count = max(label_dist.values())
    imbalance_ratio = max_count / min_count
    
    if imbalance_ratio > 10:
        print('⚠️  WARNING: Severe label imbalance detected!')
        print(f'   Majority class has {imbalance_ratio:.1f}x more samples than minority')
        print('   This may affect model performance on minority classes')
        print('   Consider adding more data for underrepresented classes')
    elif imbalance_ratio > 5:
        print('⚠️  Note: Moderate label imbalance detected')
        print(f'   Ratio: {imbalance_ratio:.1f}x between max and min')
    else:
        print('✅ Label distribution is well balanced')
    
    DATA_READY = True
else:
    print(f'❌ Preprocessed data not found')
    print(f'   Run preprocessing (Step 5) first')
    DATA_READY = False

print('')
print('=' * 60)

## Step 5: Install Dependencies

In [ ]:
%%capture
# Install Unsloth and dependencies (output captured to reduce noise)
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install -q hf_transfer
!pip install -q wandb
!pip install -q scikit-learn
!pip install -q -r requirements.txt

## Step 6: Pre-Download Model

Download model Qwen3-8B terlebih dahulu agar **bisa di-resume** jika terputus.
Jika download terhenti, **jalankan cell ini lagi** — akan lanjut dari progress terakhir.
Skip jika model sudah ter-cache dari CPT notebook.

In [ ]:
# Pre-download model dengan resume support
# Skip otomatis jika sudah ter-cache dari CPT notebook

from huggingface_hub import snapshot_download
from pathlib import Path
import shutil

MODEL_LOCAL = "/content/hf_cache/Qwen3-8B-bnb-4bit"

# Check if already cached
existing_files = list(Path(MODEL_LOCAL).rglob("*.safetensors")) if Path(MODEL_LOCAL).exists() else []
if existing_files:
    total_gb = sum(f.stat().st_size for f in existing_files) / 1e9
    print(f"✅ Model sudah ter-cache: {total_gb:.2f} GB ({len(existing_files)} safetensor files)")
    print(f"   Path: {MODEL_LOCAL}")
    print("   (Skip download)")
else:
    disk = shutil.disk_usage("/content")
    free_gb = disk.free / 1e9
    print(f"Disk tersedia: {free_gb:.1f} GB (butuh ~5 GB)")

    if free_gb < 6:
        print("⚠️  Disk hampir penuh! Pertimbangkan hapus file yang tidak perlu.")

    print(f"\nDownloading {MODEL_NAME} ke {MODEL_LOCAL}...")
    print("(Jika terputus, jalankan cell ini lagi untuk melanjutkan)\n")

    snapshot_download(
        repo_id=MODEL_NAME,
        local_dir=MODEL_LOCAL,
        resume_download=True,
    )

    files = list(Path(MODEL_LOCAL).rglob("*.safetensors"))
    total_gb = sum(f.stat().st_size for f in files) / 1e9
    print(f"\n✅ Model downloaded: {total_gb:.2f} GB ({len(files)} safetensor files)")
    print(f"   Path: {MODEL_LOCAL}")

## Step 7: Preprocess SFT Data (Run if data changed)

In [ ]:
# Preprocess SFT data
# Run this when you add new CSV/XLSX files to Dataset/SFT/

print('\n' + '=' * 60)
print('PREPROCESSING SFT DATA')
print('=' * 60)
print('This will:')
print('  1. Load all CSV/XLSX files from Dataset/SFT/')
print('  2. Convert to Alpaca format (instruction/input/output)')
print('  3. Normalize labels to DFK categories')
print('  4. Assign reasoning templates')
print('  5. Split into train/val (90/10)')
print('  6. Save to Dataset/SFT/processed/')
print('')

!python scripts/preprocess_sft.py

print('')
print('=' * 60)
print('PREPROCESSING COMPLETE')
print('=' * 60)
print('✅ Ready for training!')
print('')
print('Next step: Run training (Step 8)')

## Step 8: Configure & Run SFT Training

In [ ]:
# Run SFT training with W&B monitoring
print('\n' + '=' * 60)
print('STARTING SFT TRAINING')
print('=' * 60)
print('')
print('Configuration:')
print(f"  Base model: {MODEL_NAME}")
print(f"  CPT LoRA: {'Auto-load' if CPT_AVAILABLE else 'Not available'}")
print(f"  LoRA rank: 16")
print(f"  Batch size (effective): {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Validation split: {VALIDATION_SPLIT * 100}%")
print('')
print('Monitoring:')
print(f"  W&B project: {WANDB_PROJECT}")
print('')
print('Notes:')
print('  - Output saved to: outputs/sft/lora_adapter/')
print('  - Validation metrics tracked in W&B')
print('  - If interrupted, run this cell again')
print('')
print('Estimated time: 1-4 hours (Tesla T4)')
print('=' * 60)
print('')

# Execute training
!python scripts/train_sft.py --config configs/sft_config.yaml

## Step 9: Real-Time Training Monitoring

**Training is running. Keep this page open and check W&B dashboard.**

You can monitor the following live:

In [ ]:
# Check W&B run status
print('\n' + '=' * 60)
print('W&B RUN STATUS')
print('=' * 60)

try:
    import wandb
    
    if 'WANDB_DISABLED' not in os.environ:
        # Get current run
        api = wandb.Api()
        
        # Find recent run in SFT project
        runs = api.runs(path=WANDB_PROJECT)
        sft_runs = [r for r in runs if 'sft' in r.name.lower()]
        
        if sft_runs:
            latest_run = sft_runs[0]
            print(f'✅ W&B Run Active!')
            print(f"   Run name: {latest_run.name}")
            print(f"   State: {latest_run.state}")
            print(f"   Created: {latest_run.created_at}")
            print('')
            print(f"📊 View dashboard: {latest_run.url}")
            print('')
            print('Metrics available in W&B:')
            
            # Get history of metrics
            history = latest_run.history()
            
            for key in list(history.keys()):
                if not key.startswith('_'):
                    print(f"  • {key}")
            print('')
            print('What to watch for:')
            print('  ✓ train/loss decreasing steadily')
            print('  ✓ eval/loss should be lower than train/loss')
            print('  ✓ eval/accuracy should increase')
            print('  ✓ eval/f1_macro should increase')
            print('  ✓ No NaN or Inf values')
            print('')
            print('Signs of healthy training:')
            print('  ✓ Train loss < Eval loss (not overfitting)')
            print('  ✓ Both losses decreasing')
            print('  ✓ Accuracy converging toward final value')
            print('  ✓ F1 score stable near end of training')
            print('')
            print('Warning signs:')
            print('  ⚠ Train loss increasing')
            print('  ⚠ Eval loss > Train loss (severe overfitting)')
            print('  ⚠ Accuracy not improving')
            print('  ⚠ Loss spikes (data or learning rate issue)')
        else:
            print('ℹ️  No W&B runs found yet.')
            print('   Check back after training starts.')
            print('   Make sure W&B API key was set in Step 2')
    
except Exception as e:
    print(f'ℹ️  Could not check W&B status: {e}')
    print('   Training may be running without monitoring')

## Step 10: Run Inference Test (Optional)

**After training completes, run this cell to test the model on sample inputs.**

In [ ]:
# Inference test - Quick check on sample texts
if RUN_INFERENCE_TEST:
    print('\n' + '=' * 60)
    print('RUNNING INFERENCE TEST')
    print('=' * 60)
    
    import sys
    sys.path.insert(0, 'scripts')
    from utils import format_alpaca_prompt
    from unsloth import FastLanguageModel
    
    print('Loading model...')
    model, tokenizer = FastLanguageModel.from_pretrained(
        MODEL_NAME,
        max_seq_length=2048,
        load_in_4bit=True,
    )
    
    # Load the SFT LoRA (which already contains CPT knowledge if CPT was done first)
    sft_lora_path = 'outputs/sft/lora_adapter'
    cpt_lora_path = 'outputs/cpt/lora_adapter'
    
    import os
    from peft import PeftModel
    
    if os.path.exists(sft_lora_path):
        print('Loading SFT LoRA...')
        model = PeftModel.from_pretrained(model, sft_lora_path)
    elif os.path.exists(cpt_lora_path):
        print('SFT LoRA not found, loading CPT LoRA...')
        model = PeftModel.from_pretrained(model, cpt_lora_path)
    else:
        print('No LoRA adapters found. Using base model.')
    
    # Set to inference mode
    FastLanguageModel.for_inference(model)
    print('Model loaded!')
    
    # Test samples
    test_samples = [
        {
            "input": "Vaksin COVID-19 mengandung microchip untuk melacak pergerakan manusia. Bill Gates mendalangi ini semua.",
            "expected": "Hoax"
        },
        {
            "input": "Pemerintah Kominfo mengklarifikasi bahwa berita tentang vaksin adalah tidak benar.",
            "expected": "Bukan DFK"
        },
        {
            "input": "Orang-orang [suku] adalah penyebab semua masalah di negara ini. Mereka harus diusir!",
            "expected": "Ujaran Kebencian"
        },
        {
            "input": "Dia korup dan mencuri uang rakyat. Polisi harus menangkap dia segera.",
            "expected": "Fitnah"
        },
    ]
    
    instruction = "Klasifikasikan teks berikut apakah termasuk konten DFK (Disinformasi, Fitnah, atau Kebencian) dan berikan alasannya."
    
    print('Running inference on test samples...')
    print('')
    
    for i, sample in enumerate(test_samples):
        prompt = format_alpaca_prompt(instruction, sample["input"], "")
        
        # Tokenize
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
        
        # Generate
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
        
        # Decode
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract response part
        if "### Respons:" in response:
            prediction = response.split("### Respons:")[-1].strip()
        else:
            prediction = response.strip()
        
        # Get predicted label
        predicted_label = "Unknown"
        for label in ["Hoax", "Disinformasi", "Misinformasi", "Ujaran Kebencian", "Fitnah", "Bukan DFK"]:
            if f"Kategori: {label}" in prediction:
                predicted_label = label
                break
        
        # Check correctness
        expected = sample["expected"]
        correct = predicted_label == expected
        status = "CORRECT" if correct else "WRONG"
        
        print(f"\nTest {i+1}:")
        print(f"  Input: {sample['input'][:80]}...")
        print(f"  Expected: {expected}")
        print(f"  Predicted: {predicted_label}")
        print(f"  Status: {status}")
        print(f"  Output: {prediction[:150]}...")
    
else:
    print('Inference test disabled. Set RUN_INFERENCE_TEST = True to enable.')

print('')
print('=' * 60)
print('INFERENCE TEST COMPLETE')
print('=' * 60)

## Step 11: Verify SFT Output

In [ ]:
# Verify SFT training output
import os
from pathlib import Path

print('\n' + '=' * 60)
print('SFT TRAINING VERIFICATION')
print('=' * 60)

output_dir = Path('outputs/sft')
lora_path = output_dir / 'lora_adapter'

if lora_path.exists():
    print('✅ LoRA adapter saved successfully!')
    print(f'   Location: {lora_path}')
    print('')
    print('Files in LoRA adapter:')
    total_size = 0
    for f in sorted(lora_path.glob('*')):
        size_kb = f.stat().st_size / 1024
        total_size += size_kb
        print(f'  - {f.name} ({size_kb:.1f} KB)')
    
    print(f'   Total size: {total_size/1024:.1f} MB')
    print('')
    print('=' * 60)
    print('SFT TRAINING COMPLETE!')
    print('=' * 60)
    print('')
    print('Model Status:')
    print(f"  ✅ Base model: {MODEL_NAME}")
    print(f"  ✅ CPT LoRA: {'Loaded' if CPT_AVAILABLE else 'Not used'}")
    print(f"  ✅ SFT LoRA: Trained on top")
    print('')
    print('Final model is ready for inference!')
    print('')
    print('Future updates:')
    print('  - Add new CSV to Dataset/SFT/')
    print('  - Rerun preprocessing (Step 6)')
    print('  - Rerun training (Step 7)')
    print('  - Model will continue from this checkpoint')
    print('')
    print('W&B Monitoring:')
    print(f"  View full results: https://wandb.ai/{WANDB_ENTITY or user}/{WANDB_PROJECT}")
else:
    print('❌ LoRA adapter not found!')
    print('   Training may have failed or is incomplete.')
    print('')
    print('Troubleshooting:')
    print('  1. Check training output above for errors')
    print('  2. Verify GPU is available (nvidia-smi)')
    print('  3. Check W&B dashboard for failure logs')

## Quick Reference: W&B Metrics Guide

In [ ]:
# Print quick reference for W&B metrics interpretation
print('\n' + '=' * 60)
print('W&B METRICS INTERPRETATION GUIDE')
print('=' * 60)
print('')
print('📉 Training Metrics (train/*):')
print('   train/loss: How well model follows instructions')
print('   train/epoch: Current training epoch')
print('   train/learning_rate: Learning rate over time')
print('   train/grad_norm: Gradient magnitude (should be stable)')
print('')
print('📊 Validation Metrics (eval/*):')
print('   eval/loss: Validation loss (lower = better)')
print('   eval/accuracy: Overall classification accuracy')
print('   eval/f1_macro: Macro F1 (averages all classes equally)')
print('   eval/precision_macro: Weighted precision')
print('   eval/recall_macro: Weighted recall')
print('')
print('✅ Signs of Healthy Training:')
print('   ✓ Both train and eval loss decreasing')
print('   ✓ eval/loss < train/loss (gap not too large)')
print('   ✓ Accuracy increases then plateaus')
print('   ✓ F1, Precision, Recall all improving')
print('   ✓ Grad norm stable, no spikes')
print('')
print('⚠️  Signs of Issues:')
print('   ⚠ eval/loss > train/loss (severe overfitting)')
print('   ⚠ Accuracy not improving or decreasing')
print('   ⚠ Large gap between train and eval loss')
print('   ⚠ NaN or Inf values in metrics')
print('')
print('=' * 60)


1. **Loaded the base model** (or used cached version)
   - Model: `{MODEL_NAME}`
   - Quantized to 4-bit for efficiency

2. **Auto-loaded CPT LoRA** (if available)
   - Model already understands Indonesian DFK patterns
   - Continues from CPT checkpoint

3. **Applied SFT LoRA adapter**
   - Trained on labeled DFK classification data
   - Alpaca format: instruction/input/output
   - Validation metrics tracked in real-time

4. **Saved SFT LoRA adapter**
   - Combines CPT + SFT knowledge
   - Ready for DFK classification inference

### Output Generated
```
outputs/sft/
└── lora_adapter/
    ├── adapter_config.json
    ├── adapter_model.safetensors
    └── tokenizer.json
```

### Time Taken
- Preprocessing: 1-5 minutes
- Training: 1-4 hours
- **Total: ~1.5-4.5 hours**

### W&B Monitoring Benefits

**You can now:**
- View all training runs in W&B dashboard
- Compare different hyperparameter configurations
- Track validation metrics in real-time
- Share results with team members
- Identify issues early (overfitting, data problems)

### Incremental Updates

**When you get new SFT data:**

1. Upload new CSV/XLSX files to `Dataset/SFT/`
2. Rerun preprocessing (Step 6)
3. Rerun training (Step 7)

**What happens:**
- The script auto-loads CPT LoRA if available
- SFT trains on top of CPT-enhanced model
- New data refines classification accuracy
- W&B creates a new run for comparison
- No need to re-download or reinstall anything

**Result:** The model continues improving with each new dataset.

### Model Loading for Inference

To load the trained model for inference:

```python
from unsloth import FastLanguageModel
from peft import PeftModel

# Load base model
model, tokenizer = FastLanguageModel.from_pretrained(
    'unsloth/Qwen3-8B-bnb-4bit',  # Or your MODEL_NAME
    load_in_4bit=True,
)

# Load CPT LoRA (optional, but recommended)
model = PeftModel.from_pretrained(model, 'outputs/cpt/lora_adapter')

# Load SFT LoRA
model = PeftModel.from_pretrained(model, 'outputs/sft/lora_adapter')

# Set to inference mode
FastLanguageModel.for_inference(model)

# Now model is ready for inference!
```

### Changing Base Model

Want to use a different model? Edit the **Configuration Section** (Step 2):

```python
# Examples:
MODEL_NAME = "unsloth/Qwen3-8B-bnb-4bit"  # Qwen3 (default)
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"  # Qwen2.5
MODEL_NAME = "unsloth/Meta-Llama-3.1-8B-bnb-4bit"  # Llama 3.1
```

Then run Step 6-10 again. The model will be downloaded, cached, and trained with both CPT and SFT.